In [1]:

!pip install exifread
!pip install requests
!pip install datetime
!pip install Pillow
!pip install pillow_heif

from pillow_heif import register_heif_opener

register_heif_opener()

from PIL import Image, ImageDraw, ImageFont
import exifread
import requests
import datetime



def get_address_from_image(image_path):
  # Extract GPS data from EXIF
  r =open(image_path, 'rb')
  try: 
    tags = exifread.process_file(r)
  except:
    orientation_ref = "Rotated 90 CW"
    words = Country_folder_name.split()
    country = words[1]
    #date_created = os.path.getctime(image_path)
    #date_print = time.strftime("%m/%d/%y", time.localtime(date_created))
    address = ["", ""]
  else:
      with open(image_path, 'rb') as f:
        tags = exifread.process_file(f)
        latitude = tags.get('GPS GPSLatitude')
        longitude = tags.get('GPS GPSLongitude')
        latitude_ref = tags.get('GPS GPSLatitudeRef')
        longitude_ref = tags.get('GPS GPSLongitudeRef')
        orientation_ref = tags.get('Image Orientation')
        date_time_original = tags.get('EXIF DateTimeOriginal')

  #Tags = piexif.dump(tags)
  # Check if GPS data is available
      if not latitude or not longitude or not date_time_original:
        orientation_ref = "Rotated 90 CW"
        words = Country_folder_name.split()
        country = words[1]
        #date_created = os.path.getctime(image_path)
        #date_print = time.strftime("%m/%d/%y", time.localtime(date_created))
        address = ["", ""]
        return address, country , orientation_ref
  #print(tags)
      date_obj = datetime.strptime(str(date_time_original), "%Y:%m:%d %H:%M:%S")
      date_print = " (" + date_obj.strftime("%b %d,'%y") +")"
  #print(date_print)

  # Convert coordinates to decimal degrees
      latitude = convert_to_decimal(latitude)
      longitude = convert_to_decimal(longitude)

  # Set north/south and east/west based on reference tags
 # latitude_dir = 'N' if latitude_ref == 'N' else 'S'
 # longitude_dir = 'E' if longitude_ref == 'E' else 'W'

   # Fix + - for the latitude / longuitude
      if str(latitude_ref) == "S":
        latitude = -latitude
      if str(longitude_ref) == "W":
        longitude = -longitude

  # Use geocoding API to get address
      api_key = 'YOUR API KEY'  # Replace with your API key
      url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latitude},{longitude}&key={api_key}"
      response = requests.get(url)
      data = response.json()
      #print(data)
        # Extract and return address if successful
      # Extract address components if successful
      ##print(data['results'])
      if data['status'] == 'OK':
        city = ""
        state = ""
        county = ""
        country = "*"
        for result in data['results']:
          # Use address_components for more precise extraction
          for component in result['address_components']:
            if component['types'][0] == 'administrative_area_level_1':
              state = component['short_name']
            elif component['types'][0] == 'country':
              country = component['long_name']
            elif component['types'][0] == 'administrative_area_level_2':
              county = component['long_name']
            elif component['types'][0] == 'locality':
              city = component['long_name']
        # Use formatted_address as fallback
        if city != "":
           city = city+", "
        if state != "":
           state = state+", "
        if county != "":
           county = county+", "
        if country != "*":
           country_a = country 
        else:
           country_a = ""
        ##address = f"{city}{county}{country_a}"


         ######## CITY MORE IMPORTANT
        if country == "*":
            address = ["Flying to",""]
        if city!= "":
            address = [f"{city}",f"{country_a}{date_print}"]
        elif county != "":
            address = [f"{county}",f"{country_a}{date_print}"]
        elif state != "":
            address = [f"{state}",f"{country_a}{date_print}"]
        elif country != "*":
            address = [f"{country}",f"{date_print}"]

         ##### COUNTY MORE IMPORTANT #######
        if country == "Egypt" or country == "Indonesia":
            if county != "":
                address = [f"{county}",f"{country_a}{date_print}"]
            elif state != "":
                address = [f"{state}",f"{country_a}{date_print}"]
            elif country != "*":
                address = [f"{county}",f"{date_print}"]

             ##### STATE  MORE IMPORTANT #######   
        if country == "Indonesia":
            address=[f"{city}{county}{state}",f"{country_a}{date_print}"]
            #if state != "":
                #address = [f"{city}{state}",f"{country_a}{date_print}"]
            #elif country != "*":
                #address = [f"{city}{county}",f"{date_print}"]
        if country == "Japan" or country == "India" or country == "Chile" or country == "Australia" or country == "Philippines" or country == "Vietnam" or country == "Türkiye" or country == "Tanzania" or country == "Malaysia":
            if county != "":
                address = [f"{city}{county}",f"{country_a}{date_print}"]
            else:
                address = [f"{city}{state}",f"{country_a}{date_print}"]

        #print(f"{city}, {county}, {state}, {country} {date_print}")


      else:
        orientation_ref = "Rotated 90 CW"
        words = Country_folder_name.split()
        country = words[1]
        #date_created = os.path.getctime(image_path)
        #date_print = time.strftime("%m/%d/%y", time.localtime(date_created))
        address = ["", ""]


      #print(address)
  return address, country , orientation_ref



def convert_to_decimal(coord):
  degrees = float(coord.values[0].num) / float(coord.values[0].den)
  minutes = float(coord.values[1].num) / float(coord.values[1].den)
  seconds = float(coord.values[2].num) / float(coord.values[2].den)
  return degrees + minutes / 60 + seconds / 3600

'''
# Example usage
image_path = '/content/drive/MyDrive/Image processing/IMG_8556.JPG'
address = get_address_from_image(image_path)

if address:
  print(address)
else:
  print("No GPS data found in image or geocoding failed.")
'''

'\n# Example usage\nimage_path = \'/content/drive/MyDrive/Image processing/IMG_8556.JPG\'\naddress = get_address_from_image(image_path)\n\nif address:\n  print(address)\nelse:\n  print("No GPS data found in image or geocoding failed.")\n'

In [2]:
####### LOAD FLAG ########
def find_image_by_word(word):
  #print(word)
  for filename in os.listdir(flags_path):
    if word.lower() in filename.lower():  # Case-insensitive search
      full_path = os.path.join(flags_folder_path, filename)
      return full_path

In [3]:
from io import StringIO

################# BURNING THE ADDRESS FUNCTION #################
###################################################
#import os
#!pip install Pillow
#from PIL import Image, ImageDraw, ImageFont


# Replace placeholders with your actual paths


def burn_text(folder_path, font_path):


  ########## GET THE IMAGE DATE ###############
  # Get folder name and date
  file_name = os.path.basename(input_image_path)
  #print(folder_name)
  #print(input_image_path)
  '''
  creation_date = datetime.fromtimestamp(os.path.getctime(input_image_path))
  formatted_date = creation_date.strftime("%A, %B %dth, %Y")
  print(formatted_date)
  '''






  #############################################
  ##### CALL GET ADDRESS AND FLAG FUNCTION #######
  ##print((get_address_from_image(input_image_path)))
  if isinstance(get_address_from_image(input_image_path),tuple) :
    formatted_address, Country, Orientation = get_address_from_image(input_image_path)
    #print(formatted_address, Country)
    #print(isinstance(get_address_from_image(input_image_path),tuple))
  else:

    return None

  image = Image.open(os.path.join(folder_path,filename))

  if str(Orientation) == "Horizontal (normal)":
    font_size = 120
    resize_factor = 0.6
    lower_margin_factor = 1
  elif str(Orientation) == "Rotated 180":
    image = image.transpose(Image.ROTATE_180)
    font_size = 120
    resize_factor = 0.6
    lower_margin_factor = 1
  elif str(Orientation) == "Rotated 90 CW":
    image = image.transpose(Image.ROTATE_270)
    font_size =  120
    resize_factor = 0.7
    lower_margin_factor = 1
  elif str(Orientation) == "Rotated 90 UW":
    image = image.transpose(Image.ROTATE_90)
    font_size =  120
    resize_factor = 0.7
    lower_margin_factor = 1
  else :
    Orientation = "Rotated 90 CW"
    font_size =  120
    resize_factor = 0.5
    lower_margin_factor = 1



  #############################################
  ####### CALL find_image_by_word FUNCTION #######

  #### Exception
  if Country == "Türkiye":
    Country = "Turkey"


  '''

  full_path= find_image_by_word(Country)
  print(Country)
  png_overlay = Image.open(full_path)
  _, flag_size = png_overlay.size
  #resize_factor = 0.25
  # Resize the PNG
  #png_overlay = png_overlay.resize((int(png_overlay.width * resize_factor), int(png_overlay.height * resize_factor)))






################################
################################
########## ORIGINAL ############
################################

  img_width, img_height = image.size
  # MArgin
  margin = 20 # 5 pixels by default
########## TEXT & MATT############

### DEFINE TEXT ####
   ##### CENTER TEXT ######
    
  #print(formatted_address)  
  if len(str(formatted_address[0])) > len(str(formatted_address[1])):
    size = len(str(formatted_address[0]))
  else:
    size = len(str(formatted_address[1]))

  centered = [line.center(size) for line in formatted_address]
  cent_address = '\n'.join(centered)

    
    
    
  # Calculate text bbox with margin adjustment
  draw = ImageDraw.Draw(image)
  font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
  text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)
  #print(Orientation)
  #### If text its too long, reduce the text font####
  if str(Orientation) == "Rotated 90 CW" or str(Orientation) == "Rotated 90 UW":
        #print((text_bbox[2]+margin*20+flag_size),(img_height*9/16))
        #print(font_size)
        if (text_bbox[2]+margin*20+flag_size) > (img_height*9/16):
        # Reduce Fint Size until conditions are met
 
          while True: 
              font_size = font_size -1
              font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
              text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)
              if (text_bbox[2]+margin*20+flag_size) <= (img_height*9/16):
                break
        #print((text_bbox[2]+margin*20+flag_size),(img_height*9/16))
        #print(font_size)
  else:
    if (text_bbox[2]+margin*20) > img_width:
        if (text_bbox[2]+margin*20) > (img_height*9/16):
        # Reduce Fint Size until conditions are met
 
          while True: 
              font_size = font_size -1
              font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
              text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)
              if (text_bbox[2]+margin*20) <= (img_height*9/16):
                break      
    
        
       # font_size = int(font_size*(img_width)//(text_bbox[2]+margin*20))
       # font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
       # text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)
        
    
  # Define Size fo the Flag

  resize_factor =text_bbox[3]/ flag_size
  png_overlay = png_overlay.resize((int(png_overlay.width * resize_factor), int(png_overlay.height * resize_factor)))


  # Calculate position based on bbox with additional margin
  x_pos = img_width //2 - text_bbox[2]//2 +(png_overlay.size[0])//4 ## - margin*5  # right edge + margin
  y_pos = img_height - text_bbox[3] - margin*3  # bottom edge + margin


#### DEFINE MAT #######
# Define box attributes
  box_opacity =230 ### 0 to 255
  box_radius = 35  # Adjust the corner radius
  box_color = (255, 0, 0, box_opacity)  # Red box (customize the color as needed)
  box_width = text_bbox[2]   # Adjust the width
  box_height = text_bbox[3]  # Adjust the height
  box_x = x_pos  # Center the box horizontally
  box_y = y_pos  # Place the box on top (customize the position)



  country_color= {"Argentina": [(45, 154, 227, box_opacity),(255, 255, 255)],
               "Australia": [(16, 106, 224, box_opacity),(255, 255, 255)],
               "Brazil": [(41, 179, 34, box_opacity),(255, 255, 255)],
               "Cambodia": [(232, 82, 242, box_opacity),(255, 255, 255)],
               "Chile": [(219, 15, 124, box_opacity),(255, 255, 255)],
               "Colombia": [(227, 174, 30, box_opacity),(255, 255, 255)],
               "Costa Rica": [(70, 105, 189, box_opacity),(255, 255, 255)],
               "Egypt": [(173, 134, 24, box_opacity),(255, 255, 255)],
               "Hong Kong": [(235, 53, 40, box_opacity),(255, 255, 255)],
               "India": [(201, 196, 26, box_opacity),(255, 255, 255)],
               "Indonesia":[ (81, 191, 31, box_opacity),(255, 255, 255)],
               "Japan": [(242, 53, 31, box_opacity),(255, 255, 255)],
               "Malaysia": [(222, 88, 27, box_opacity),(255, 255, 255)],
               "Morocco": [(250, 27, 65, box_opacity),(255, 255, 255)],
               "New Zealand": [(3, 125, 173, box_opacity),(255, 255, 255)],
               "Peru":[ (207, 112, 116, box_opacity),(255, 255, 255)],
               "Philippines": [(224, 187, 0, box_opacity),(255, 255, 255)],
               "South Africa": [(163, 83, 18, box_opacity),(255, 255, 255)],
               "Singapore": [(186, 22, 52, box_opacity),(255, 255, 255)],
               "Tanzania": [(17, 189, 94, box_opacity),(255, 255, 255)],
               "Turkey": [(224, 45, 224, box_opacity),(255, 255, 255)],
               "United States":[(29, 135, 181, box_opacity),(255, 255, 255)],
               "Vietnam": [(173, 24, 143, box_opacity),(255, 255, 255)],
               "Patagonia": [(240, 240, 242, box_opacity),(255, 255, 255)],
               "*": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Qatar": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Thailand": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Switzerland": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Fiji": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Kenya": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "United Arab Emirates": [(115, 117, 115, box_opacity),(255, 255, 255)],
               "Macao": [(115, 117, 115, box_opacity),(255, 255, 255)]
               }


   # Access value by key
  #print(country_color["Turkey"])
   # Access value by key
  #print(country_color["Turkey
  box_color = country_color[Country][0]
  font_color = country_color[Country][1]
  
  #try:
   # box_color = country_color[Country][0]
    #font_color = country_color[Country][1]
  #finally:
   # box_color = (115, 117, 115, box_opacity)
    #font_color = (255, 255, 255)
  

# Define the rounded rectangle coordinates
  left_top_x = box_x + box_radius
  left_top_y = box_y + box_radius
  right_top_x = box_x + box_width - box_radius
  right_top_y = box_y + box_radius
  right_bottom_x = box_x + box_width - box_radius
  right_bottom_y = box_y + box_height - box_radius
  left_bottom_x = box_x + box_radius
  left_bottom_y = box_y + box_height - box_radius

# Create a transparent overlay image
  overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))  # Transparent black background
  draw = ImageDraw.Draw(overlay)

############ DRAW MAT ##############

# Draw the colored box on the overlay
  draw.rounded_rectangle((box_x - 4*margin, box_y, box_x + box_width + 3*margin, box_y + box_height),radius=box_radius, fill=box_color)
  image.paste(overlay, (0,0), mask=overlay.convert("RGBA").split()[3])



########## DEFINE TEXT AGAIN TO MAKE SURE IT BURNS ###########

  # Calculate text bbox with margin adjustment
  draw = ImageDraw.Draw(image)
  font = ImageFont.truetype(font_path, font_size)  # Adjust font size and path
  text_bbox = draw.textbbox((0, 0), str(cent_address), font=font)

  # Calculate position based on bbox with additional margin
  x_pos = img_width //2 - text_bbox[2]//2 +(png_overlay.size[0])//4 #- margin*5  # right edge + margin
  y_pos = img_height - text_bbox[3] - margin*3    # bottom edge + margin

  #cent_address = "   dfd \n   ddd"
############ DRAW TEXT ##############
  # Draw text with justification and margin
  draw.text((x_pos, y_pos), str(cent_address), font=font, fill=font_color) # White text






############## FLAG ###########


  # Position the overlay image (adjust coordinates as needed)
  #resize_factor =text_bbox[3]/ flag_size
  #png_overlay = png_overlay.resize((int(png_overlay.width * resize_factor), int(png_overlay.height * resize_factor)))

  overlay_x, overlay_y = (x_pos - png_overlay.width -margin//4,  y_pos) ## img_height - png_overlay.width//2) # bottom edge + margin

  # Paste the overlay onto the background with transparency
  image.paste(png_overlay, (overlay_x, overlay_y), mask=png_overlay.convert("RGBA").split()[3])
  '''
  # Copy EXIF
  
  exif = image.info['exif']
    
  
  output_path_date_country = os.path.join(output_path_date,Country + "/" )
  if not os.path.exists(output_path_date_country):
    os.makedirs(output_path_date_country)
    #print(f"Folder '{output_path_date_country}' created successfully!")
  #else:
    #print(f"Folder '{output_path_date_country}' already exists.")  
  
  if formatted_address[0] == "Aoraki / Mount Cook, ":
    formatted_address[0] = "Aoraki - Mount Cook, "
  if formatted_address[0] == "Franz Josef / Waiau, ":
    formatted_address[0] = "Franz Josef - Waiau,"   

  


  output_path = os.path.join(output_path_date_country, formatted_address[0] + ".jpg") 
  #print(output_path)
  #print(os.path.exists(output_path))
  if os.path.exists(output_path):
    output_path_new = output_path = os.path.join(output_path_date_country, formatted_address[0]) 
    while True: 
        output_path_new = (output_path_new + " ") 
        #print("1." +output_path_new)
        if not os.path.exists((output_path_new +".jpg")):
            break
  else:
    output_path_new = os.path.join(output_path_date_country, formatted_address[0])
    #print("2." +output_path_new)
    output_path_new = output_path_new + ".jpg"

    
    
  #print(output_path_new)
  if str(Orientation) == "Horizontal (normal)":
    image.save(output_path_new,'JPEG',quality=100)
  else:
    if str(Orientation) == "Rotated 180":
        image=image.rotate(180, expand=True)
        image.save(output_path_new,'JPEG',quality=100, exif = exif)
    elif str(Orientation) == "Rotated 90 CW":

        image=image.rotate(90, expand=True)
        image.save(output_path_new,'JPEG',quality=100, exif = exif)
    elif str(Orientation) == "Rotated 90 UW":

        image=image.rotate(270, expand=True)
        image.save(output_path_new,'JPEG',quality=100, exif = exif)
    else :

        image=image.rotate(0, expand=True)
        image.save(output_path_new,'JPEG',quality=100, exif = exif)

  if output_path_new[-1] == " ":
    os.rename(output_path_new,output_path_new + ".jpg")
  
  #image.save('/content/drive/MyDrive/Image processing/02 USA/destination.jpg', 'JPEG', exif=exif) 
  # Save image
  #image = image.rotate(90)



In [6]:
################################################################################################
################################################################################################
################################################################################################
# Iterate through each JPG file in the directory

import os
from pathlib import Path
from datetime import datetime
Country_folder_name = "22. India"
desired_aspect_ratio = "16:9"
matte_color = (255, 245, 238)
shade_opacity = 30
#flags_path = flags_path
#folder_path = date_input_filepath
font_path = "/Users/josemarimon/Desktop/Frame Python/Fonts/Courier_Prime/CourierPrime-BoldItalic.ttf"  # Choose a suitable font
date_input_filepath = Path("/Users/josemarimon/Desktop/Frame Python/Original Pics/"+(Country_folder_name)+"/")
output_path_date = "/Users/josemarimon/Desktop/Frame Python/Edited/Clean/"
##matt_image_directory = output_path_date
##output_path_matt = "/Users/josemarimon/Desktop/Frame Python/Fotos eileentu/2024-02-18-08h17/Takeout/Google Photos/"+(Country_folder_name)+"/With Matt"
flags_path = "/Users/josemarimon/Desktop/Frame Python/Flags/"
flags_folder_path = os.path.dirname(flags_path)

#################
#################
i=0
#################
#################
file_count = 0
for root, directories, files in os.walk(date_input_filepath):
    for file in files:
        file_count += 1

file_list = []
for file_name1 in os.listdir(date_input_filepath):
    file_list.append(file_name1)
file_list.sort(key=lambda file_name1: file_name1.lower())
#print(file_list)



for filename in file_list:
  i += 1 

  if filename.endswith(".JPG") or filename.endswith(".jpg") or filename.endswith(".heic") or filename.endswith(".HEIC"):
 
    input_image_path = os.path.join(date_input_filepath, filename)
    print(input_image_path)

    burn_text(date_input_filepath,font_path)
    #input_image_path_matt = os.path.join(output_path_date, "dated_" + filename)
    #output_image_path_matt = os.path.join(output_path_matt, "date_matted_" + filename)
    ##add_mat_with_shade(input_image_path_matt, output_image_path_matt, desired_aspect_ratio, matte_color, shade_opacity)
    print("Mat added to " +filename +" " + str(i) + "/" +str(file_count) +" " + str(round((i)/file_count*100,1))+"%")

    



/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2125.JPG
Mat added to IMG_2125.JPG 2/867 0.2%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2128.JPG
Mat added to IMG_2128.JPG 3/867 0.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2129.JPG
Mat added to IMG_2129.JPG 4/867 0.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2130.JPG
Mat added to IMG_2130.JPG 5/867 0.6%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2132.JPG
Mat added to IMG_2132.JPG 6/867 0.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2133.JPG
Mat added to IMG_2133.JPG 7/867 0.8%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2135.JPG
Mat added to IMG_2135.JPG 8/867 0.9%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2136.JPG
Mat added to IMG_2136.JPG 9/867 1.0%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2138.JPG
Mat added t

Mat added to IMG_2288.JPG 73/867 8.4%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2290.JPG
Mat added to IMG_2290.JPG 74/867 8.5%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2292.JPG
Mat added to IMG_2292.JPG 75/867 8.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2293.JPG
Mat added to IMG_2293.JPG 76/867 8.8%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2295.JPG
Mat added to IMG_2295.JPG 77/867 8.9%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2298.JPG
Mat added to IMG_2298.JPG 78/867 9.0%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2303.JPG
Mat added to IMG_2303.JPG 79/867 9.1%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2305.JPG
Mat added to IMG_2305.JPG 80/867 9.2%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2307.JPG
Mat added to IMG_2307.JPG 81/867 9.3%
/Users/josemarimon/Desktop/Frame Python/Or

Mat added to IMG_2420.JPG 144/867 16.6%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2422.JPG
Mat added to IMG_2422.JPG 145/867 16.7%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2424.JPG
Mat added to IMG_2424.JPG 146/867 16.8%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2425.JPG
Mat added to IMG_2425.JPG 147/867 17.0%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2426.JPG
Mat added to IMG_2426.JPG 148/867 17.1%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2427.JPG
Mat added to IMG_2427.JPG 149/867 17.2%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2433.JPG
Mat added to IMG_2433.JPG 150/867 17.3%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2435.JPG
Mat added to IMG_2435.JPG 151/867 17.4%
/Users/josemarimon/Desktop/Frame Python/Original Pics/22. India/IMG_2438.JPG
Mat added to IMG_2438.JPG 152/867 17.5%
/Users/josemarimon/Deskt